In [33]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import json


In [34]:
nltk.download('punkt', quiet=True);
nltk.download('stopwords', quiet=True);
nltk.download('wordnet', quiet=True);
nltk.download('omw-1.4', quiet=True);
nltk.download('punkt_tab', quiet=True);


# Structured data Preprocessing 

In [35]:
structing = pd.DataFrame(pd.read_csv('nyt-ingredients-snapshot-2015.csv'))
structing = structing.dropna(subset=['input']).reset_index(drop=True)
print(structing.info())
structing

<class 'pandas.DataFrame'>
RangeIndex: 179063 entries, 0 to 179062
Data columns (total 7 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   index      179063 non-null  int64  
 1   input      179063 non-null  str    
 2   name       178668 non-null  str    
 3   qty        179063 non-null  float64
 4   range_end  173842 non-null  float64
 5   unit       123009 non-null  str    
 6   comment    98617 non-null   str    
dtypes: float64(2), int64(1), str(4)
memory usage: 19.8 MB
None


,index,input,name,qty,range_end,unit,comment
0,0,1 1/4 cups cooked and pureed fresh butternut s...,butternut squash,1.25,0.0,cup,"cooked and pureed fresh, or 1 10-ounce package..."
1,1,1 cup peeled and cooked fresh chestnuts (about...,chestnuts,1.00,0.0,cup,"peeled and cooked fresh (about 20), or 1 cup c..."
2,2,"1 medium-size onion, peeled and chopped",onion,1.00,0.0,NaN,"medium-size, peeled and chopped"
3,3,"2 stalks celery, chopped coarse",celery,2.00,0.0,stalk,chopped coarse
4,4,1 1/2 tablespoons vegetable oil,vegetable oil,1.50,0.0,tablespoon,NaN
...,...,...,...,...,...,...,...
179058,179202,3/4 oz. pineapple juice,pineapple juice,0.75,0.0,ounce,NaN
179059,179203,1 tsp. fresh lemon juice,lemon juice,1.00,0.0,teaspoon,fresh
179060,179204,Angostura bitters,Angostura bitters,0.00,0.0,NaN,NaN
179061,179205,Wedge of pineapple,pineapple,1.00,0.0,wedge,NaN


In [36]:
print(structing.input[0])
structing.comment[0]

1 1/4 cups cooked and pureed fresh butternut squash, or 1 10-ounce package frozen squash, defrosted


'cooked and pureed fresh, or 1 10-ounce package frozen squash, defrosted'

redundant words set

In [37]:
with open('measures_english.json', 'r') as file:
    measure = json.load(file)

measures = pd.DataFrame(measure["measures"])
measure = []
for i in measures.forms:
    measure += i
measure = set(measure)
print(list(measure)[:10])

['mm', 'p', 'dash', 'jar', 'smidgeon', 'fl qt', 'q.', 'pc.', 'ds.', 'cms.']


In [38]:
names = set(structing['name'].copy().tolist())
print(list(names)[:10])

['basic focaccia recipe', 'orange Cura&#231;ao', 'Parma prosciutto ham', 'peas or cooked fresh peas', 'red chilie flakes', 'Pecorino', 'sherry vinegar, champagne vinegar or red wine vinegar', 'ground chipotle chilies', 'white popcorn', 'tripe']


In [40]:
def red(text):

    # Tokenization
    
    tokens = word_tokenize(text)

    # Convert to lowercase
    tokens = [token.lower() for token in tokens]

    # Remove punctuation and non-alphanumeric characters
    tokens = [token for token in tokens if token.isalpha()]

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]

    # Stemming
    lemmer = WordNetLemmatizer()

    tokens = [lemmer.lemmatize(token) for token in tokens]

    # Remove units
    tokens_ = [token for token in tokens if token not in measure]

    # Remove 1, 2 and 3 worded food names
    tokens = []
    for i in range(len(tokens_)-2):
        if tokens_[i] not in names and " ".join([tokens_[i], tokens_[i+1]]) not in names and " ".join([tokens_[i], tokens_[i+1], tokens_[i+2]]) not in names:
            tokens.append(tokens_[i])

    return tokens

In [41]:
comments = structing.comment.copy()
comments = comments.dropna().reset_index(drop=True)
comments = comments.apply(red)
redund = []
for i in comments:
    if i:
        redund += i
redund = set(redund)
print(list(measure)[:20])

['mm', 'p', 'dash', 'jar', 'smidgeon', 'fl qt', 'q.', 'pc.', 'ds.', 'cms.', 'pkgs.', 'oz', 'boxed', 'gill', 'gram', 'litre', 'milliliter', 'drop', 'milligramme', 'kgr']


In [42]:
if "refrigerated" in redund:
    print("true")

else:
    print("false")

true


FUNCTIONS FOR PREPROCESSING

In [43]:
import re

def manipulate_string(input_string):
    # Remove special characters except for specific cases
    #modified_string = re.sub(r'[^a-zA-Z0-9*%&+/]', '', input_string)
    
    # Replace % if preceded by a number
    # modified_string = re.sub(r'(\d)%', r'\1%', input_string)

    # Remove % is not preceded by a number
    modified_string = re.sub(r'(?<!\d)%', '', input_string)
    
    # Replace & with 'and'
    modified_string = modified_string.replace('&', ' and ')
    
    # Replace + with 'plus'
    modified_string = modified_string.replace('+', ' plus ')
    
    # Replace / if it is between digits
    modified_string = re.sub(r"(?<![0-9])/|/(?![0-9])", " ", modified_string)
    
    # Replace - with 'to' if it is between digits
    modified_string = re.sub(r'(\d)-(\d)', r'\1 to \2', modified_string)
    modified_string = re.sub(r"(?<![0-9])-|-(?![0-9])", " ", modified_string)

    # Remove brackets
    modified_string = re.sub(r'[\(\)\[\]\{\}]', '', modified_string)
    
    return modified_string


In [44]:
english_stopwords = stopwords.words('english')

if "bottle" in english_stopwords:
    print("'' is included in the list of English stopwords.")
else:
    print("'packages' is not included in the list of English stopwords.")

'packages' is not included in the list of English stopwords.


In [45]:
def preprocess(text):

    # Replace for special cases
    text = text.replace('half a', '1/2')
    text = text.replace('half of', '1/2')
    text = text.replace('half from', '1/2')

    # Special character removal
    text = manipulate_string(text)
    
    # Tokenization
    tokens = word_tokenize(text)

    # join back %
    modified_list = []
    for i, item in enumerate(tokens):
        if i > 0 and item == "%":
            modified_list[-1] += item
        else:
            modified_list.append(item)
    
    # Convert to lowercase
    tokens = [token.lower() for token in modified_list]
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    stop_words.remove('to')
    stop_words.remove('and')
    stop_words.remove('can')
    tokens = [token for token in tokens if token not in stop_words]
    tokens = [token for token in tokens if token not in redund]


    # Lemmatizing
    lemmer = WordNetLemmatizer()

    tokens = [lemmer.lemmatize(token) for token in tokens]

    return tokens

In [46]:
lemmer1 = WordNetLemmatizer()
print(lemmer1.lemmatize('m'))

m


Testing

In [47]:
for i in range(20):
    print(preprocess(structing.input[i]))

['1', '1/4', 'cup', 'and', 'butternut', 'squash', ',', '1', '10', 'ounce', 'package', 'squash', ',']
['1', 'cup', 'and', 'chestnut', '20', ',', '1', 'cup', 'canned', ',', 'chestnut']
['1', 'onion', ',', 'and', 'chopped']
['2', 'stalk', 'celery', ',', 'chopped']
['1', '1/2', 'tablespoon', 'vegetable', 'oil']
['2', 'tablespoon', 'gelatin', ',', '1/2', 'cup', 'water']
['salt']
['1', 'cup', 'canned', 'plum', 'tomato', 'juice']
['6', 'cup', 'veal', 'beef', 'stock']
['1/3', 'cup', 'worcestershire', 'sauce']
['1', 'tablespoon', 'sauce']
['1/2', 'teaspoon', 'pepper', 'flake']
['4', 'bay', 'leaf']
['6', 'clove', 'garlic', ',', 'and', 'chopped']
['2', 'carrot', ',', 'and', 'diced']
['2', 'onion', ',', 'diced']
['6', 'tablespoon', 'butter']
['1', 'tablespoon', 'seasoning', ',', 'seasoning']
['3', 'pound', 'beef', 'brisket']
['1/2', 'cup', 'bread', 'crumb']


In [48]:
for i in range(20):
    arr = preprocess(structing.input[i])
            
    int_mul = 0
    mul_idx = 0
    
    int_frac = 0
    start_frac_idx = 0
    end_frac_idx = 0
    
    final_string = ''
    for m in range(len(arr)):        
        if '/' in arr[m]:
            try:
                int_frac = int(arr[m-1]) + (int(arr[m].split('/')[0]) / int(arr[m].split('/')[1]))   
                start_frac_idx = m-1
                end_frac_idx = m
            except Exception as e:
                int_frac = int(arr[m].split('/')[0]) / int(arr[m].split('/')[1])
                start_frac_idx = m
                end_frac_idx = m
                pass
        try:
            int_mul = int(arr[m-1]) * int(arr[m])
        except Exception as e:
            pass
        
        if (int_frac != 0):
            final_string = final_string[:len(final_string)-2] + '{:.2f}'.format(int_frac) + final_string[len(final_string)+1:] + ' ' 
            int_frac = 0                
        elif (int_mul != 0):
            final_string = final_string[:len(final_string)-2] + str(int_mul) + final_string[len(final_string)+1:] + ' ' 
            int_mul = 0
        else:
            final_string += arr[m] + ' '        
    print(final_string)

1.25 cup and butternut squash , 10 ounce package squash , 
1 cup and chestnut 20 , 1 cup canned , chestnut 
1 onion , and chopped 
2 stalk celery , chopped 
1.50 tablespoon vegetable oil 
2 tablespoon gelatin 0.50 cup water 
salt 
1 cup canned plum tomato juice 
6 cup veal beef stock 
0.33 cup worcestershire sauce 
1 tablespoon sauce 
0.50 teaspoon pepper flake 
4 bay leaf 
6 clove garlic , and chopped 
2 carrot , and diced 
2 onion , diced 
6 tablespoon butter 
1 tablespoon seasoning , seasoning 
3 pound beef brisket 
0.50 cup bread crumb 


The Real Thing

In [49]:
structing['input_pre'] = structing['input'].apply(preprocess)
structing

,index,input,name,qty,range_end,unit,comment,input_pre
0,0,1 1/4 cups cooked and pureed fresh butternut s...,butternut squash,1.25,0.0,cup,"cooked and pureed fresh, or 1 10-ounce package...","[1, 1/4, cup, and, butternut, squash, ,, 1, 10..."
1,1,1 cup peeled and cooked fresh chestnuts (about...,chestnuts,1.00,0.0,cup,"peeled and cooked fresh (about 20), or 1 cup c...","[1, cup, and, chestnut, 20, ,, 1, cup, canned,..."
2,2,"1 medium-size onion, peeled and chopped",onion,1.00,0.0,NaN,"medium-size, peeled and chopped","[1, onion, ,, and, chopped]"
3,3,"2 stalks celery, chopped coarse",celery,2.00,0.0,stalk,chopped coarse,"[2, stalk, celery, ,, chopped]"
4,4,1 1/2 tablespoons vegetable oil,vegetable oil,1.50,0.0,tablespoon,NaN,"[1, 1/2, tablespoon, vegetable, oil]"
...,...,...,...,...,...,...,...,...
179058,179202,3/4 oz. pineapple juice,pineapple juice,0.75,0.0,ounce,NaN,"[3/4, oz, ., pineapple, juice]"
179059,179203,1 tsp. fresh lemon juice,lemon juice,1.00,0.0,teaspoon,fresh,"[1, tsp, ., lemon, juice]"
179060,179204,Angostura bitters,Angostura bitters,0.00,0.0,NaN,NaN,"[angostura, bitter]"
179061,179205,Wedge of pineapple,pineapple,1.00,0.0,wedge,NaN,[pineapple]


Extracting measurement units

In [50]:
structing['input_pre'][363]

['1', '6', 'ounce', 'can', 'tomato', 'paste']

In [51]:
def extract_units(structing,measure):
    extract={}
    measure_dic={"jar","can","packet","package","box","bottle","container", "jars","cans","packets","packages","boxes","bottles","containers","jarred","canned","packaged","bottled","conatined","boxed"}
    count = {}
    texts=structing['input_pre']
    
    start = 0
    flag = True

    for k in range(len(texts)):
        for a in range(len(texts[k])):
            
            if texts[k][a] in measure and texts[k][a] != "": 
                extract[a] = texts[k][a]
               
        
        ele = ''
        for i in extract:
            if extract[i] in measure_dic:
                continue
            else:
                flag = False
                ele = extract[i]
                break
                
                
        if flag == False:
            count = 0
            for i in extract:
                #print("before ", texts[k])
                texts[k].pop(i-count)
                count = count +1
                #print("after ", texts[k])
                
          
            texts[k].insert(list(extract.keys())[0] , ele)
                
        
        flag= True
        extract = {}
        #print(texts[k])
    return texts

In [52]:
text=extract_units(structing,measure)
structing['input_pre']= text
print(structing['input_pre'])

0         [1, 1/4, cup, and, butternut, squash, ,, 1, 10...
1            [1, cup, and, chestnut, 20, ,, 1, ,, chestnut]
2                               [1, onion, ,, and, chopped]
3                            [2, stalk, celery, ,, chopped]
4                      [1, 1/2, tablespoon, vegetable, oil]
                                ...                        
179058                       [3/4, oz, ., pineapple, juice]
179059                            [1, tsp, ., lemon, juice]
179060                                  [angostura, bitter]
179061                                          [pineapple]
179062                                             [cherry]
Name: input_pre, Length: 179063, dtype: object


– ”clove” and ”garlic”;

– ”stick” and ”butter”/”margarine”/”cinnamon”/ ”car-rot”/”celery”;

– ”sprig” and ”rosemary”/”thyme”/”mint”/”parsley”; – ”link” and ”sausage”;

– ”stalk” and ”celery”/”green onion”/”spring onion”/”broccoli”/”kale”/”cauliflower”;

– ”sheet” and ”gelatin”;

– ”cube” and ”stock”/”butter”,”margarine”;

– ”head” and ”cabbage”/”lettuce”/”cauliflower”;

In [53]:
for i in range(20):
    arr = preprocess(structing.input[i])
            
    int_mul = 0
    mul_idx = 0
    
    int_frac = 0
    start_frac_idx = 0
    end_frac_idx = 0
    
    measurements = []  # List to store extracted measurements
    
    final_string = ''
    for m in range(len(arr)):
        # Check if the element is a fraction
        if '/' in arr[m]:
            try:
                int_frac = int(arr[m-1]) + (int(arr[m].split('/')[0]) / int(arr[m].split('/')[1]))   
                start_frac_idx = m-1
                end_frac_idx = m
            except Exception as e:
                int_frac = int(arr[m].split('/')[0]) / int(arr[m].split('/')[1])
                start_frac_idx = m
                end_frac_idx = m
                pass
        
        # Check if the elements form a multiplication
        try:
            int_mul = int(arr[m-1]) * int(arr[m])
        except Exception as e:
            pass
        
        # Check if the element is a measurement unit
        if arr[m] in measure:
            measurements.append( arr[m])
        
        # Generate the final string
        if int_frac != 0:
            final_string = final_string[:len(final_string)-2] + '{:.2f}'.format(int_frac) + final_string[len(final_string)+1:] + ' ' 
            int_frac = 0                
        elif int_mul != 0:
            final_string = final_string[:len(final_string)-2] + str(int_mul) + final_string[len(final_string)+1:] + ' ' 
            int_mul = 0
        else:
            final_string += arr[m] + ' '
    
    print("Final String:", final_string)
    print("Measurements:", measurements)



Final String: 1.25 cup and butternut squash , 10 ounce package squash , 
Measurements: ['cup', 'ounce', 'package']
Final String: 1 cup and chestnut 20 , 1 cup canned , chestnut 
Measurements: ['cup', 'cup', 'canned']
Final String: 1 onion , and chopped 
Measurements: []
Final String: 2 stalk celery , chopped 
Measurements: ['stalk']
Final String: 1.50 tablespoon vegetable oil 
Measurements: ['tablespoon']
Final String: 2 tablespoon gelatin 0.50 cup water 
Measurements: ['tablespoon', 'cup']
Final String: salt 
Measurements: []
Final String: 1 cup canned plum tomato juice 
Measurements: ['cup', 'canned']
Final String: 6 cup veal beef stock 
Measurements: ['cup']
Final String: 0.33 cup worcestershire sauce 
Measurements: ['cup']
Final String: 1 tablespoon sauce 
Measurements: ['tablespoon']
Final String: 0.50 teaspoon pepper flake 
Measurements: ['teaspoon']
Final String: 4 bay leaf 
Measurements: ['leaf']
Final String: 6 clove garlic , and chopped 
Measurements: ['clove']
Final String: 

In [54]:

for i in range(20):
    arr = preprocess(structing.input[i])
            
    int_mul = 0
    mul_idx = 0
    
    int_frac = 0
    start_frac_idx = 0
    end_frac_idx = 0
    
    measurements = []  # List to store extracted measurements
    
    final_string = ''
    for m in range(len(arr)):
        # Check if the element is a fraction
        if '/' in arr[m]:
            try:
                int_frac = int(arr[m-1]) + (int(arr[m].split('/')[0]) / int(arr[m].split('/')[1]))   
                start_frac_idx = m-1
                end_frac_idx = m
            except Exception as e:
                int_frac = int(arr[m].split('/')[0]) / int(arr[m].split('/')[1])
                start_frac_idx = m
                end_frac_idx = m
                pass
        
        # Check if the elements form a multiplication
        try:
            int_mul = int(arr[m-1]) * int(arr[m])
        except Exception as e:
            pass
        
        # Check if the element is a measurement unit
        if arr[m] in measure:
            measurements.append(arr[m])
        
        # Generate the final string
        if int_frac != 0:
            final_string = final_string[:len(final_string)-2] + '{:.2f}'.format(int_frac) + final_string[len(final_string)+1:] + ' ' 
            int_frac = 0                
        elif int_mul != 0:
            final_string = final_string[:len(final_string)-2] + str(int_mul) + final_string[len(final_string)+1:] + ' ' 
            int_mul = 0
        else:
            final_string += arr[m] + ' '
    
    # Extract the food item
    food_item = final_string.lower()
    for word in redund:
        if word in food_item.split():
            food_item = food_item.replace(word, '')
    
    # Special cases for certain measurement units considered as food items
    special_units = {"jar","can","packet","package","box","bottle","container", "jars","cans","packets","packages","boxes","bottles","containers","canned","bottled","packed","packaged","jarred","boxed","contained"}
    for unit in special_units:
        if unit in food_item.split():
            if len(measurements) == 0 or any(unit in m for m in measurements):
                food_item = food_item.replace(unit, '')
            else:
                measurements.append(unit)
    
    food_item = ' '.join(food_item.split())  # Remove extra whitespace
    
    print("Final String:", final_string)
    print("Measurements:", measurements)
    print("Food Item:", food_item.strip())








Final String: 1.25 cup and butternut squash , 10 ounce package squash , 
Measurements: ['cup', 'ounce', 'package']
Food Item: 1.25 cup and butternut squash , 10 ounce squash ,
Final String: 1 cup and chestnut 20 , 1 cup canned , chestnut 
Measurements: ['cup', 'cup', 'canned']
Food Item: 1 cup and chestnut 20 , 1 cup , chestnut
Final String: 1 onion , and chopped 
Measurements: []
Food Item: 1 onion , and chopped
Final String: 2 stalk celery , chopped 
Measurements: ['stalk']
Food Item: 2 stalk celery , chopped
Final String: 1.50 tablespoon vegetable oil 
Measurements: ['tablespoon']
Food Item: 1.50 tablespoon vegetable oil
Final String: 2 tablespoon gelatin 0.50 cup water 
Measurements: ['tablespoon', 'cup']
Food Item: 2 tablespoon gelatin 0.50 cup water
Final String: salt 
Measurements: []
Food Item: salt
Final String: 1 cup canned plum tomato juice 
Measurements: ['cup', 'canned']
Food Item: 1 cup plum tomato juice
Final String: 6 cup veal beef stock 
Measurements: ['cup']
Food Item

In [55]:

for i in range(20):
    arr = preprocess(structing.input[i])
            
    int_mul = 0
    mul_idx = 0
    
    int_frac = 0
    start_frac_idx = 0
    end_frac_idx = 0
    
    measurements = []  # List to store extracted measurements
    quantity=''
    
    final_string = ''
    for m in range(len(arr)):
        # Check if the element is a fraction
        if '/' in arr[m]:
            try:
                int_frac = int(arr[m-1]) + (int(arr[m].split('/')[0]) / int(arr[m].split('/')[1]))   
                start_frac_idx = m-1
                end_frac_idx = m
            except Exception as e:
                int_frac = int(arr[m].split('/')[0]) / int(arr[m].split('/')[1])
                start_frac_idx = m
                end_frac_idx = m
                pass
        
        # Check if the elements form a multiplication
        try:
            int_mul = int(arr[m-1]) * int(arr[m])
        except Exception as e:
            pass
        split_string = final_string.split()

        # Check if the element is a measurement unit
        if arr[m] in measure:
            measurements.append(arr[m])
        if len(split_string) > 1:
            unit = split_string[1]

        
        # Generate the final string
        if int_frac != 0:
            final_string = final_string[:len(final_string)-2] + '{:.2f}'.format(int_frac) + final_string[len(final_string)+1:] + ' ' 
            int_frac = 0                
        elif int_mul != 0:
            final_string = final_string[:len(final_string)-2] + str(int_mul) + final_string[len(final_string)+1:] + ' ' 
            int_mul = 0
        else:
            final_string += arr[m] + ' '
    
    # Extract the food item
    food_item = final_string.lower()
    for word in redund:
        if word in food_item.split():
            food_item = food_item.replace(word, '')
    
    # Special cases for certain measurement units considered as food items
    special_units = {"jar","can","packet","package","box","bottle","container", "jars","cans","packets","packages","boxes","bottles","containers","canned","bottled","packed","packaged","jarred","boxed","contained"}
    for unit in special_units:
        if unit in food_item.split():
            if len(measurements) == 0 or any(unit in m for m in measurements):
                food_item = food_item.replace(unit, '')
            else:
                measurements.append(unit)
    
    #food_item = ' '.join(food_item.split())  # Remove extra whitespace
    
    # Extracting exactly one quantity, one unit of measurement, and one food item
    unit = measurements.pop(0) if len(measurements) > 0 else ""
    words = final_string.split()

    if words[0].replace(".", "").isdigit():
        quantity = words[0]

    # Remove the quantity from the line
    food_item = final_string.replace(quantity, "")

# Remove leading and trailing whitespace
    food_item = food_item.strip()
    if len(split_string) > 2:
        food_item = " ".join(split_string[2:])


    
    print("Final String:", final_string)
    print("Quantity:", quantity)
    print("Unit:", unit)
    print("Food Item:", food_item)
    print()


    # Define the CSV file path
    



Final String: 1.25 cup and butternut squash , 10 ounce package squash , 
Quantity: 1.25
Unit: cup
Food Item: and butternut squash , 10 ounce package squash

Final String: 1 cup and chestnut 20 , 1 cup canned , chestnut 
Quantity: 1
Unit: cup
Food Item: and chestnut 20 , 1 cup canned ,

Final String: 1 onion , and chopped 
Quantity: 1
Unit: 
Food Item: , and

Final String: 2 stalk celery , chopped 
Quantity: 2
Unit: stalk
Food Item: celery ,

Final String: 1.50 tablespoon vegetable oil 
Quantity: 1.50
Unit: tablespoon
Food Item: vegetable

Final String: 2 tablespoon gelatin 0.50 cup water 
Quantity: 2
Unit: tablespoon
Food Item: gelatin 0.50 cup

Final String: salt 
Quantity: 
Unit: 
Food Item: salt

Final String: 1 cup canned plum tomato juice 
Quantity: 1
Unit: cup
Food Item: canned plum tomato

Final String: 6 cup veal beef stock 
Quantity: 6
Unit: cup
Food Item: veal beef

Final String: 0.33 cup worcestershire sauce 
Quantity: 0.33
Unit: cup
Food Item: worcestershire

Final String: 

In [56]:
import csv

# Create a CSV file and write the headers
with open('extracted_data.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Final String', 'Quantity', 'Unit', 'Food Item'])

    for i in range(20):
        arr = preprocess(structing.input[i])
                
        int_mul = 0
        mul_idx = 0
        
        int_frac = 0
        start_frac_idx = 0
        end_frac_idx = 0
        
        measurements = []  # List to store extracted measurements
        quantity = ''
        
        final_string = ''
        for m in range(len(arr)):
            # Check if the element is a fraction
            if '/' in arr[m]:
                try:
                    int_frac = int(arr[m-1]) + (int(arr[m].split('/')[0]) / int(arr[m].split('/')[1]))   
                    start_frac_idx = m-1
                    end_frac_idx = m
                except Exception as e:
                    int_frac = int(arr[m].split('/')[0]) / int(arr[m].split('/')[1])
                    start_frac_idx = m
                    end_frac_idx = m
                    pass
            
            # Check if the elements form a multiplication
            try:
                int_mul = int(arr[m-1]) * int(arr[m])
            except Exception as e:
                pass
            split_string = final_string.split()
    
            # Check if the element is a measurement unit
            if arr[m] in measure:
                measurements.append(arr[m])
            if len(split_string) > 1:
                unit = split_string[1]
    
            
            # Generate the final string
            if int_frac != 0:
                final_string = final_string[:len(final_string)-2] + '{:.2f}'.format(int_frac) + final_string[len(final_string)+1:] + ' ' 
                int_frac = 0                
            elif int_mul != 0:
                final_string = final_string[:len(final_string)-2] + str(int_mul) + final_string[len(final_string)+1:] + ' ' 
                int_mul = 0
            else:
                final_string += arr[m] + ' '
        
        # Extract the food item
        food_item = final_string.lower()
        for word in redund:
            if word in food_item.split():
                food_item = food_item.replace(word, '')
        
        # Special cases for certain measurement units considered as food items
        special_units = {"jar","can","packet","package","box","bottle","container", "jars","cans","packets","packages","boxes","bottles","containers","canned","bottled","packed","packaged","jarred","boxed","contained"}
        for unit in special_units:
            if unit in food_item.split():
                if len(measurements) == 0 or any(unit in m for m in measurements):
                    food_item = food_item.replace(unit, '')
                else:
                    measurements.append(unit)
        
        #food_item = ' '.join(food_item.split())  # Remove extra whitespace
        
        # Extracting exactly one quantity, one unit of measurement, and one food item
        unit = measurements.pop(0) if len(measurements) > 0 else ""
        words = final_string.split()
    
        if words[0].replace(".", "").isdigit():
            quantity = words[0]
    
        # Remove the quantity from the line
        food_item = final_string.replace(quantity, "")
    
        # Remove leading and trailing whitespace
        food_item = food_item.strip()
        if len(split_string) > 2:
            food_item = " ".join(split_string[2:])
        
        # Write the extracted data to the CSV file
        writer.writerow([final_string, quantity, unit, food_item])

# Print a message indicating the successful creation of the CSV file
print("CSV file 'extracted_data.csv' has been created.")


CSV file 'extracted_data.csv' has been created.
